In [1]:
import sys
import torch
import importlib.util

print("Python:", sys.executable)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", importlib.util.find_spec("vllm"))
print("OpenAI:", importlib.util.find_spec("openai"))

Python: /home/ubuntu/llm_venv/bin/python
CUDA: True
GPU: Tesla T4
vLLM: ModuleSpec(name='vllm', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f016a18fe50>, origin='/home/ubuntu/llm_venv/lib/python3.10/site-packages/vllm/__init__.py', submodule_search_locations=['/home/ubuntu/llm_venv/lib/python3.10/site-packages/vllm'])
OpenAI: ModuleSpec(name='openai', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f016a18faf0>, origin='/home/ubuntu/llm_venv/lib/python3.10/site-packages/openai/__init__.py', submodule_search_locations=['/home/ubuntu/llm_venv/lib/python3.10/site-packages/openai'])


# Создание клиента

In [1]:
from openai import AsyncOpenAI

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

client = AsyncOpenAI(
    api_key="EMPTY",
    base_url="http://localhost:8000/v1",
)

In [2]:
response = await client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "user", "content": "Ответь кратко: какое слово считается самым смешным?"}
    ],
    temperature=0.7,
)

print(response.choices[0].message.content)

"Смешной" само по себе не является словом, считаемым самым смешным. Это слово относится к понятию комичности.


# Загрузка входных данных

In [3]:
import json

with open("../data/dev.jsonl", "r", encoding="utf-8") as f:
    dev_data = [json.loads(line) for line in f]

len(dev_data)

40

# Системный промпт

In [4]:
system_prompt = """
Ты генерируешь карточку товара для маркетплейса на основе входных данных. 
На вход ты получаешь информацию о товаре в JSON формате.

Твоя задача:
Верни только JSON с полями: description, pros, cons, tags.
Поле description должно содержать краткое описание товара в 2-5 предложениях (строка от 120 до 700 символов).
Поле pros должно описывать "плюсы" товара (список из 2-5 непустых строк).
Поле cons должно описывать "минусы" товара (список из 1-3 непустых строк).
Поле tags должно содержать короткие теги для поиска товара (список из 3-8 непустых строк).

Требования:
1. Не добавляй новые поля.
2. Не придумывай факты, которых нет во входных данных. 
3. Не раздувай текст: пиши емко и соблюдай указанные диапазоны. Лучше написать меньше, но по делу.
3. Поле description формируй только на основе title, category и attributes. Другие источники использовать запрещено.
4. Если отзыв в reviews противоречит attributes, не используй его как источник для других полей.
5. Каждый пункт pros и cons должен быть уникальным и коротким, в среднем 2-5 слов.
6. Не включай в pros или cons нейтральные факты.
7. В tags сочетай тип товара, назначение и ключевые характеристики; каждый тег должен быть полезен для поиска.

Пример обработки:
Вход:
{
  "product_id": "P-1001",
  "title": "Беспроводные наушники AirBass X2",
  "category": "Электроника / Наушники",
  "attributes": {
    "бренд": "AirBass",
    "тип": "внутриканальные",
    "время работы": "24 ч",
    "Bluetooth": "5.3",
    "влагозащита": "IPX4"
  },
  "reviews": [
    "Звук чистый, бас глубокий, за свои деньги отлично",
    "На морозе быстро садятся, зимой неудобно"
  ]
}

Выход:
{
  "description": "Беспроводные внутриканальные наушники AirBass X2 работают по Bluetooth 5.3 и защищены от брызг по стандарту IPX4. Заявленное время работы составляет до 24 часов.",
  "pros": ["Чистый звук", "Глубокий бас"],
  "cons": ["Быстро разряжаются на морозе"],
  "tags": ["наушники", "беспроводные", "внутриканальные", "Bluetooth 5.3", "IPX4"]
}

А теперь обработай этот вход:
"""

# Описание контракта с помощью Pydantic

In [5]:
from pydantic import(
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
    ValidationError,
)

In [6]:
import re

def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()])

In [7]:
class ProductCard(BaseModel):
    """Результат обработки карточки."""

    model_config = ConfigDict(extra="forbid")

    product_id: str
    description: str = Field(min_length=120, max_length=700)
    pros: list[str] = Field(min_length=2, max_length=5)
    cons: list[str] = Field(min_length=1, max_length=3)
    tags: list[str] = Field(min_length=3, max_length=8)

    @field_validator("product_id", "description", mode="before")
    @classmethod
    def strip_string(cls, value):
        if isinstance(value, str):
            return value.strip()
        return value

    @field_validator("pros", "cons", "tags", mode="before")
    @classmethod
    def strip_list_items(cls, value):
        if isinstance(value, list):
            return [
                item.strip() if isinstance(item, str) else item
                for item in value
            ]
        return value

    @field_validator("product_id", "description")
    @classmethod
    def string_must_not_be_empty(cls, value: str) -> str:
        if not value:
            raise ValueError(
                "Строка не должна быть пустой | поля: product_id, description."
            )
        return value

    @field_validator("pros", "cons", "tags")
    @classmethod
    def list_items_must_not_be_empty(
        cls,
        value: list[str],
    ) -> list[str]:
        if any(not item for item in value):
            raise ValueError(
                "Элемент списка не должен быть пустым | поля: pros, cons, tags."
            )
        return value

    @field_validator("description")
    @classmethod
    def description_sentence_count(cls, value: str) -> str:
        sentence_count = count_sentences(value)

        if not 2 <= sentence_count <= 5:
            raise ValueError(
                "description должно содержать от 2 до 5 предложений"
            )

        return value

# Парсинг и валидация

## Политика обработки ошибок

- **Исправить кодом**, если ошибка детерминированная и не требует нового содержания (`product_id`, лишние поля, пробелы, дубликаты).
- **Повторить генерацию**, если нарушены требования к содержимому или структуре карточки.
- **Fallback**, если после нескольких попыток валидную карточку получить не удалось: сохранить вход, ответ и причины ошибок для ручного разбора.

In [8]:
from enum import Enum

class ValidationReason(str, Enum):
    """
    Категории ошибок в ответе модели.
    PRODUCT_ID_MISMATCH и EXTRA_FIELD обрабатываются детерминированно.
    """
    EMPTY_RESPONSE = "empty_response"
    INVALID_JSON = "invalid_json"
    MISSING_FIELD = "missing_field"
    WRONG_TYPE = "wrong_type"
    DESCRIPTION_LENGTH = "description_length"
    DESCRIPTION_SENTENCE_COUNT = "description_sentence_count"
    WRONG_LIST_LENGTH = "wrong_list_length"
    EMPTY_VALUE = "empty_value"
    LONG_INPUT = "long_input"
    OUTPUT_TRUNCATED = "output_truncated"
    OTHER = "other_validation_error"

In [9]:
class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""
    def __init__(
            self, 
            reasons: list[ValidationReason], 
            message: str,
            details: dict | None = None,
    ):
        super().__init__(message)
        self.reasons = reasons
        self.details = details or {}

In [10]:
def get_validation_info(
    error: ValidationError,
) -> tuple[list[ValidationReason], dict]:
    """Преобразует ошибки Pydantic в категории ошибок карточки."""

    reasons = []
    details = {}

    for err in error.errors():
        error_type = err["type"]
        field = err["loc"][0] if err["loc"] else None
        message = err["msg"].lower()
        input_value = err.get("input")

        if error_type == "missing":
            reason = ValidationReason.MISSING_FIELD

        elif error_type in {"string_type", "list_type"}:
            reason = ValidationReason.WRONG_TYPE

        elif (
            field == "description"
            and error_type in {"string_too_short", "string_too_long"}
        ):
            reason = ValidationReason.DESCRIPTION_LENGTH
            
            if isinstance(input_value, str):
                details["description_length"] = len(input_value.strip())

        elif (
            field in {"pros", "cons", "tags"}
            and error_type in {"too_short", "too_long"}
        ):
            reason = ValidationReason.WRONG_LIST_LENGTH

            if isinstance(input_value, list):
                details[f"{field}_length"] = len(input_value)

        elif "пуст" in message:
            reason = ValidationReason.EMPTY_VALUE

        elif (
            field == "description"
            and "предложен" in message
        ):
            reason = ValidationReason.DESCRIPTION_SENTENCE_COUNT

            if isinstance(input_value, str):
                details["description_sentence_count"] = count_sentences(
                    input_value
                )

        else:
            reason = ValidationReason.OTHER

        reasons.append(reason)

    reasons = list(dict.fromkeys(reasons))

    return reasons, details

In [11]:
def repair_response(data: dict, source_product: dict) -> dict:
    """Вносит детерминированные исправления в сгенерированную карточку."""

    allowed_fields = {
        "product_id", "description", "pros", "cons", "tags"
    }

    repaired = {
        key : value
        for key, value in data.items()
        if key in allowed_fields
    }

    repaired["product_id"] = source_product["product_id"]

    return repaired



In [12]:
import json

def parse_and_validate(content: str, source_product: dict) -> ProductCard:
    """Распарсить JSON-ответ, проверить контракт карточки и исправить ответ."""
    if not content or not content.strip():
        raise ResponseValidationError(
            reasons=[ValidationReason.EMPTY_RESPONSE],
            message="Модель вернула пустой ответ.",
        )

    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        raise ResponseValidationError(
            reasons=[ValidationReason.INVALID_JSON],
            message="Ответ модели не является корректным JSON.",
        ) from error

    if not isinstance(data, dict):
        raise ResponseValidationError(
            reasons=[ValidationReason.WRONG_TYPE],
            message="Ответ модели должен быть JSON-объектом.",
        )

    repaired = repair_response(data, source_product)

    try:
        return ProductCard.model_validate(repaired)

    except ValidationError as error:
        reasons, details = get_validation_info(error)

        raise ResponseValidationError(
            reasons=reasons,
            message=str(error),
            details=details,
        ) from error

# Input и output генерации

In [13]:
def build_messages(source_product: dict) -> list[dict]:
    return [
        {
            "role" : "system",
            "content" : system_prompt,
        },
        {
            "role" : "user",
            "content" : json.dumps(source_product, ensure_ascii=False)
        }
    ]

In [14]:
def build_retry_message(error: ResponseValidationError) -> str:
    """Формирует инструкцию для повторной генерации карточки."""
    fixes = []
    details = error.details or {}

    if ValidationReason.INVALID_JSON in error.reasons:
        fixes.append(
            "Верни один корректный JSON-объект без текста до или после него."
        )

    if ValidationReason.MISSING_FIELD in error.reasons:
        fixes.append(
            "JSON должен содержать все поля: description, pros, cons, tags."
        )

    if ValidationReason.WRONG_TYPE in error.reasons:
        fixes.append(
            "description должен быть строкой, "
            "pros, cons и tags - списками строк."
        )

    if ValidationReason.DESCRIPTION_LENGTH in error.reasons:
        length = details.get("description_length")

        if length is not None:
            if length < 120:
                fixes.append(
                    f"description содержит только {length} символов. "
                    "Необходимо не менее 120 символов. Добавь побольше деталей."
                )
            elif length > 700:
                fixes.append(
                    f"description содержит {length} символов - это слишком много. "
                    "Нельзя использовать более 700 символов. Сокрти текст."
                )

    if ValidationReason.DESCRIPTION_SENTENCE_COUNT in error.reasons:
        sentence_count = details.get("description_sentence_count")

        if sentence_count is not None:
            if sentence_count < 2:
                fixes.append(
                    f"description содержит {sentence_count} предложений - "
                    "это слишком мало. Напиши еще одно предложение."
                )
            elif sentence_count > 5:
                fixes.append(
                    f"description содержит {sentence_count} предложений - "
                    "а нужно не более 5 предложений. Убери лишние предложения."
                )

    if ValidationReason.WRONG_LIST_LENGTH in error.reasons:
        targets = {
            "pros": (2, 5),
            "cons": (1, 3),
            "tags": (3, 8),
        }

        for field, (min_length, target_max) in targets.items():
            length = details.get(f"{field}_length")

            if length is None:
                continue

            if length < min_length:
                fixes.append(
                    f"{field} содержит {length} элементов — это слишком мало. "
                    f"Добавь только недостающие элементы: "
                    f"в итоге должно быть {min_length}-{target_max}."
                )

            elif length > target_max:
                fixes.append(
                    f"{field} содержит {length} элементов — это слишком много. "
                    f"Удаляй лишние элементы и оставь только "
                    f"{min_length}-{target_max} самых подходящих."
                )

    if ValidationReason.EMPTY_VALUE in error.reasons:
        fixes.append(
            "Не используй пустые строки или пустые элементы списков."
        )

    if ValidationReason.OUTPUT_TRUNCATED in error.reasons:
        fixes.append(
            "Предыдущий ответ был обрезан. "
            "Сделай ответ существенно короче."
        )

    if ValidationReason.OTHER in error.reasons:
        fixes.append(
            "Исправь ответ в соответствии с требованиями."
        )

    return (
        "Предыдущий ответ не прошёл валидацию. "
        "Сгенерируй весь JSON заново.\n"
        "КРИТИЧЕСКИ ВАЖНО: не увеличивай объём ответа без необходимости. "
        "Если чего-то слишком много, удаляй лишнее, а не добавляй новый текст.\n"
        "Исправь нарушения:\n- "
        + "\n- ".join(fixes)
    )

In [15]:
class GenerationResult(BaseModel):
    """Результат генерации одной карточки."""

    product_id: str
    card: ProductCard | None
    valid: bool
    attempts: int
    errors: list[ValidationReason]
    input_tokens: int = 0
    output_tokens: int = 0

# Асинхронная генерация

In [16]:
from openai import AsyncOpenAI, BadRequestError

async def generate_one(
        client: AsyncOpenAI,
        model_name: str,
        source_product: dict,
        generation_params: dict,
        max_attempts: int = 3,
        diagnostic_dir: str | None = None,
) -> GenerationResult:
    """Генерирует и валидирует одну карточку с повторными попытками."""
    api_params = generation_params.copy()

    base_seed = api_params.pop("seed", 42)

    extra_body = {}
    for name in ("top_k", "repetition_penalty"):
        if name in api_params:
            extra_body[name] = api_params.pop(name)
    

    messages = build_messages(source_product)

    all_errors = []
    input_tokens = 0
    output_tokens = 0
    attempts_log = []

    for attempt in range(1, max_attempts + 1):
        attempt_seed = base_seed + attempt - 1

        attempt_log = {
            "attempt": attempt,
            "seed": attempt_seed,
            "messages": [message.copy() for message in messages],
        }

        try:
            response = await client.chat.completions.create(
                model=model_name,
                messages=messages,
                response_format={"type": "json_object"},
                extra_body=extra_body,
                seed=attempt_seed,
                **api_params,
            )

        except BadRequestError as error:
            error_text = str(error)

            if "maximum context length" in error_text:
                attempt_log["api_error"] = error_text
                attempt_log["validation_reasons"] = [
                    ValidationReason.LONG_INPUT.value
                ]
                attempt_log["validation_details"] = {}

                attempts_log.append(attempt_log)

                if diagnostic_dir is not None:
                    save_failure_diagnostic(
                        diagnostic_dir=diagnostic_dir,
                        source_product=source_product,
                        generation_params=generation_params,
                        attempts_log=attempts_log,
                    )

                return GenerationResult(
                    product_id=source_product["product_id"],
                    card=None,
                    valid=False,
                    attempts=attempt,
                    errors=all_errors + [ValidationReason.LONG_INPUT],
                    input_tokens=input_tokens,
                    output_tokens=output_tokens,
                )

            raise

        content = response.choices[0].message.content
        finish_reason = response.choices[0].finish_reason

        attempt_log["raw_response"] = content
        attempt_log["finish_reason"] = finish_reason

        if response.usage is not None:
            input_tokens += response.usage.prompt_tokens or 0
            output_tokens += response.usage.completion_tokens or 0

        try:
            if finish_reason == "length":
                raise ResponseValidationError(
                    reasons=[ValidationReason.OUTPUT_TRUNCATED],
                    message="Ответ модели был обрезан по max_tokens.",
                )
                
            card = parse_and_validate(
                content=content,
                source_product=source_product
            )  

            return GenerationResult(
                product_id=source_product["product_id"],
                card=card,
                valid=True,
                attempts=attempt,
                errors=all_errors,
                input_tokens=input_tokens,
                output_tokens=output_tokens,
            )
        
        except ResponseValidationError as error:
            attempt_log["validation_reasons"] = [
                reason.value for reason in error.reasons
            ]

            attempt_log["validation_details"] = error.details
            attempts_log.append(attempt_log)

            all_errors.extend(error.reasons)
            
            if attempt == max_attempts:
                break 

            if ValidationReason.EMPTY_RESPONSE in error.reasons:
                messages = build_messages(source_product)
                continue

            messages = build_messages(source_product)

            if ValidationReason.OUTPUT_TRUNCATED in error.reasons:
                messages.append(
                    {
                        "role": "user",
                        "content": build_retry_message(error),
                    }
                )
            else:
                messages.extend([
                    {
                        "role": "assistant",
                        "content": content,
                    },
                    {
                        "role": "user",
                        "content": build_retry_message(error),
                    },
                ])

    if diagnostic_dir is not None:
        save_failure_diagnostic(
            diagnostic_dir=diagnostic_dir,
            source_product=source_product,
            generation_params=generation_params,
            attempts_log=attempts_log,
        )

    return GenerationResult(
        product_id=source_product["product_id"],
        card=None,
        valid=False,
        attempts=max_attempts,
        errors=all_errors,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
    )

# Функции для сохранения результатов

In [17]:
from pathlib import Path

def load_checkpoint(path: str) -> set[str]:
    """Возвращает product_id уже обработанных товаров."""

    path = Path(path)

    if not path.exists():
        return set()

    with path.open("r", encoding="utf-8") as f:
        return {
            json.loads(line)["product_id"]
            for line in f
            if line.strip()
        }


def save_checkpoint(path: str, result: GenerationResult) -> None:
    """Добавляет результат обработки товара в checkpoint."""

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(result.model_dump_json() + "\n")


def load_results(path: str) -> list[GenerationResult]:
    """Загружает все результаты генерации из checkpoint."""

    path = Path(path)

    if not path.exists():
        return []

    with path.open("r", encoding="utf-8") as f:
        return [
            GenerationResult.model_validate_json(line)
            for line in f
            if line.strip()
        ]

In [18]:
def save_failure_diagnostic(
    diagnostic_dir: str,
    source_product: dict,
    generation_params: dict,
    attempts_log: list[dict],
) -> None:
    """Сохраняет подробную диагностику карточки, не прошедшей все попытки."""

    diagnostic_dir = Path(diagnostic_dir)
    diagnostic_dir.mkdir(parents=True, exist_ok=True)

    temperature = str(
        generation_params.get("temperature", "none")
    ).replace(".", "")

    top_p = str(
        generation_params.get("top_p", "none")
    ).replace(".", "")

    filename = (
        f"{source_product['product_id']}"
        f"_temp_{temperature}"
        f"_top_p_{top_p}.json"
    )

    data = {
        "product_id": source_product["product_id"],
        "generation_params": generation_params,
        "attempts": attempts_log,
    }

    with (diagnostic_dir / filename).open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

In [19]:
def save_experiment_result(path: str, result: dict) -> None:
    """Добавляет компактный итог одного эксперимента в текстовый файл."""

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    line = (
        f"temperature={result['temperature']}, "
        f"top_p={result['top_p']} | "
        f"valid_rate={result['valid_rate']:.3f} | "
        f"avg_attempts={result['avg_attempts']:.2f}"
    )

    with path.open("a", encoding="utf-8") as f:
        f.write(line + "\n")

# Запуск на небольшом датасете

In [20]:
async def run_dev(
    client: AsyncOpenAI,
    model_name: str,
    products: list[dict],
    generation_params: dict,
    checkpoint_path: str,
    diagnostic_dir: str | None = None,
    max_attempts: int = 3,
) -> list[GenerationResult]:
    """Генерирует карточки для dev-набора с сохранением промежуточных результатов."""

    completed_ids = load_checkpoint(checkpoint_path)

    for product in products:
        if product["product_id"] in completed_ids:
            continue

        result = await generate_one(
            client=client,
            model_name=model_name,
            source_product=product,
            generation_params=generation_params,
            max_attempts=max_attempts,
            diagnostic_dir=diagnostic_dir,
        )

        save_checkpoint(checkpoint_path, result)

    return load_results(checkpoint_path)

In [21]:
async def run_experiment(
    client: AsyncOpenAI,
    model_name: str,
    products: list[dict],
    generation_params: dict,
    checkpoint_path: str,
    diagnostic_dir: str | None = None,
    max_attempts: int = 3,
) -> dict:
    """Запускает один эксперимент и рассчитывает итоговые метрики."""

    results = await run_dev(
        client=client,
        model_name=model_name,
        products=products,
        generation_params=generation_params,
        checkpoint_path=checkpoint_path,
        diagnostic_dir=diagnostic_dir,
        max_attempts=max_attempts,
    )

    valid_count = sum(result.valid for result in results)

    return {
        **generation_params,
        "valid_rate": valid_count / len(results),
        "avg_attempts": sum(
            result.attempts for result in results
        ) / len(results),
    }

In [ ]:
temperatures = [0.4]
top_p = 0.9

temperature_results = []

for temperature in temperatures:
    temperature_label = str(temperature).replace(".", "")
    top_p_label = str(top_p).replace(".", "")

    checkpoint_path = (
        "../outputs/experiments/temperature/cache/"
        f"temp_{temperature_label}_top_p_{top_p_label}.jsonl"
    )

    diagnostic_dir = (
        "../outputs/experiments/temperature/diagnostics/"
    )

    result = await run_experiment(
        client=client,
        model_name=model_name,
        products=dev_data,
        generation_params={
            "temperature": temperature,
            "top_p": top_p,
            "max_tokens": 450,
            "seed": 43,
        },
        checkpoint_path=checkpoint_path,
        diagnostic_dir=diagnostic_dir,
        max_attempts=3,
    )

    temperature_results.append(result)

    save_experiment_result(
        path="../outputs/experiments/temperature/results.txt",
        result=result,
    )

    print(
        f"temperature={temperature} | "
        f"top_p={top_p} | "
        f"valid_rate={result['valid_rate']:.3f} | "
        f"avg_attempts={result['avg_attempts']:.2f}"
    )